# RetailMind AI — Day 3: Hyperparameter Tuning & Bias-Variance Diagnostics
**Objective**: Perform 5-fold GridSearchCV across candidate models, analyze train vs. validation gaps, and lock the best generalising model.

In [ ]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, recall_score

In [ ]:
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
df = df.dropna(subset=['TotalCharges']).copy()
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'str']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
])

In [ ]:
pipe_rf = Pipeline([('preprocessor', preprocessor), ('clf', RandomForestClassifier(random_state=42))])
param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [6, 10],
    'clf__class_weight': ['balanced', 'balanced_subsample']
}
search = GridSearchCV(pipe_rf, param_grid, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring='roc_auc', return_train_score=True, n_jobs=-1)
search.fit(X_train, y_train)

print("Best Parameters:", search.best_params_)
print("Best 5-Fold CV AUC:", round(search.best_score_, 4))
joblib.dump(search.best_estimator_, '../models/churn_v3.joblib')
print("Saved models/churn_v3.joblib")